# Patent DPR Colab Pro Training

Run this notebook on a Colab GPU runtime or a VSCode Colab-connected kernel.

This version is for the original patent retrieval experiment:

1. Use the existing `data/patent` split generated from `patent_rawdata.csv`.
2. Train DPR with BM25 hard negatives and `gold patent_id` positives.
3. Train DPR with `batch_size=128` and `epoch=40` on A100/L4 when available.


## Runtime Contract

- Drive path: `/content/drive/MyDrive/HYU-DPR/`
- Bundle: `HYU-DPR_colab_input.tar.gz`
- Data: `/content/HYU-DPR/data/patent/`
- Training output: `/content/drive/MyDrive/HYU-DPR/outputs/patent_dpr_colab_pro/`
- Evaluation basis: `gold patent_id` hit rate, not answer exact match
- Pseudo answers from `AI요약(...)` columns are kept for qualitative examples/report text only
- A100/L4 target: actual `batch_size=128`, `gradient_accumulation_steps=1`, `epoch=40`
- T4 fallback: `batch_size=64, grad_accum=2` or `batch_size=32, grad_accum=4`, still `epoch=40`

This notebook keeps the original patent retrieval setup: the positive label is the source patent ID, and a retrieval is correct when any passage chunk from that patent appears in Top-k.


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/HYU-DPR')
BUNDLE = DRIVE_ROOT / 'HYU-DPR_colab_input.tar.gz'
WORKSPACE = Path('/content/HYU-DPR')

print('DRIVE_ROOT =', DRIVE_ROOT)
print('BUNDLE =', BUNDLE)
print('WORKSPACE =', WORKSPACE)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# The notebook expects /content/drive/MyDrive/HYU-DPR/HYU-DPR_colab_input.tar.gz.
# If the file was uploaded to a different Drive folder, search MyDrive and reuse that path.
if not BUNDLE.exists():
    candidates = list(Path('/content/drive/MyDrive').rglob('HYU-DPR_colab_input.tar.gz'))
    if candidates:
        BUNDLE = candidates[0]
        DRIVE_ROOT = BUNDLE.parent
        print('Found bundle at:', BUNDLE)
        print('Using DRIVE_ROOT:', DRIVE_ROOT)
    else:
        print('Expected path:', BUNDLE)
        print('No HYU-DPR_colab_input.tar.gz found under /content/drive/MyDrive')
        print('Top-level MyDrive files/folders:')
        for p in sorted(Path('/content/drive/MyDrive').iterdir())[:50]:
            print(' -', p)
        raise FileNotFoundError('Upload HYU-DPR_colab_input.tar.gz into the mounted Google account, preferably MyDrive/HYU-DPR/.')

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Bundle size MB:', round(BUNDLE.stat().st_size / 1024 / 1024, 2))

In [ ]:
import shutil
import tarfile

if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)
WORKSPACE.mkdir(parents=True, exist_ok=True)

with tarfile.open(BUNDLE, 'r:gz') as tar:
    tar.extractall(WORKSPACE)

print('Extracted to:', WORKSPACE)
print('DPR train script:', (WORKSPACE / 'DPR-main' / 'train_dense_encoder.py').exists())
print('Patent train data:', (WORKSPACE / 'data' / 'patent' / 'train.json').exists())

## Install DPR Dependencies

In [ ]:
%cd /content/HYU-DPR/DPR-main
!python -m pip install --upgrade pip setuptools wheel
!python -m pip install "numpy<2" "transformers>=4.3" "hydra-core>=1.0.0" "omegaconf>=2.0.1" faiss-cpu jsonlines soundfile editdistance wget "spacy>=2.1.8"
!python -m pip install -e .

In [ ]:
import os
import subprocess
import sys

os.chdir("/content/HYU-DPR")
sys.path.insert(0, "/content/HYU-DPR/DPR-main")

import torch
import transformers
import dpr

print("cwd", os.getcwd())
print("torch", torch.__version__)
print("cuda", torch.cuda.is_available())
print("transformers", transformers.__version__)
print("dpr import ok")
subprocess.run(["nvidia-smi"], check=False)
subprocess.run([sys.executable, "colab/colab_runner.py", "gpu"], check=True)

## Smoke Test

This is a tiny end-to-end run on `data/patent`. It verifies data loading, DPR import, training, checkpoint writing, and `run_metadata.json` creation.


In [ ]:
%cd /content/HYU-DPR
!python colab/colab_runner.py smoke


## Batch Feasibility Check

Run this before the full training job. `auto` chooses `128` for A100/L4 and `64` otherwise.


In [ ]:
%cd /content/HYU-DPR
!python colab/colab_runner.py profile-check --profile auto --steps 20


If the profile check fails due to OOM, run fallback checks.

In [ ]:
%cd /content/HYU-DPR
# !python colab/colab_runner.py profile-check --profile 64 --steps 20
# !python colab/colab_runner.py profile-check --profile 32 --steps 20


## Full Training

Run this after the batch feasibility check succeeds. For A100/L4, this uses the DPR-paper-style global batch 128 condition on a single GPU.


In [ ]:
!nvidia-smi

In [ ]:
# If nvidia-smi shows a stale Python process from an interrupted run,
# restart the Colab runtime instead of killing an arbitrary PID.
!nvidia-smi


In [ ]:
%cd /content/HYU-DPR
!python colab/colab_runner.py train --drive-root /content/drive/MyDrive/HYU-DPR --output-name patent_dpr_colab_pro --profile 128 --epochs 40


In [ ]:
%cd /content/HYU-DPR
# Use this only when the GPU is not A100/L4, or when profile 128 fails.
# !python colab/colab_runner.py train --drive-root /content/drive/MyDrive/HYU-DPR --output-name patent_dpr_colab_pro --profile auto --auto-fallback --epochs 40


## Required Artifacts

After training, these files should exist in Google Drive:

- `/content/drive/MyDrive/HYU-DPR/outputs/patent_dpr_colab_pro/dpr_biencoder.0`
- `/content/drive/MyDrive/HYU-DPR/outputs/patent_dpr_colab_pro/train_dense_encoder.log`
- `/content/drive/MyDrive/HYU-DPR/outputs/patent_dpr_colab_pro/run_metadata.json`

After the files are copied into the local repo as `outputs/patent_dpr_colab_pro/`, local Codex can generate embeddings, run dense retrieval, evaluate `gold patent_id` Top-k accuracy, and update the report.


In [ ]:
from pathlib import Path
import json

out = Path('/content/drive/MyDrive/HYU-DPR/outputs/patent_dpr_colab_pro')
print('out:', out)
for name in ['dpr_biencoder.0', 'train_dense_encoder.log', 'run_metadata.json']:
    path = out / name
    print(name, path.exists(), round(path.stat().st_size / 1024 / 1024, 2) if path.exists() else None)

metadata_path = out / 'run_metadata.json'
if metadata_path.exists():
    print(json.dumps(json.loads(metadata_path.read_text()), indent=2, ensure_ascii=False))

manifest = Path('/content/HYU-DPR/data/patent/split_manifest.json')
if manifest.exists():
    print('split_manifest:')
    print(manifest.read_text())
